<a href="https://colab.research.google.com/github/suwarnalatha-m/Task-3-Dashboard/blob/main/task3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install dash plotly pandas jupyter-dash openpyxl

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!pip install jupyter-dash==0.4.2 dash==2.14.2

In [ ]:
# =========================
# INSTALL REQUIRED LIBRARIES
# =========================

!pip install dash
!pip install jupyter-dash
!pip install plotly
!pip install openpyxl

# =========================
# IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np

# Interactive Visualizations
import plotly.express as px
import plotly.graph_objects as go

# Dashboard Framework
from jupyter_dash import JupyterDash
from dash import Dash, dcc, html, Input, Output
from dash import Dash, dcc, html, Input, Output

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd

# Read Excel file
df = pd.read_excel(
    "Online Retail.xlsx",
    engine="openpyxl"
)

# Show dataset
print("Dataset Shape:", df.shape)

df.head()

In [ ]:
# Remove missing values
df.dropna(inplace=True)

# Remove negative values
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

# Convert date column
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Create TotalPrice column
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# Create Month column
df['Month'] = df['InvoiceDate'].dt.strftime('%b')

print("Cleaned Dataset Shape:", df.shape)

In [ ]:
import plotly.express as px
from dash import dcc, html, Input, Output
from jupyter_dash import JupyterDash

In [ ]:
app = JupyterDash(__name__)

In [ ]:
app.layout = html.Div([

    html.H1(
        "Online Retail Dashboard",
        style={
            'textAlign': 'center',
            'color': '#1F3C88'
        }
    ),

    html.Br(),

    # FILTER
    dcc.Dropdown(
        id='country_filter',
        options=[
            {'label': i, 'value': i}
            for i in df['Country'].unique()
        ],
        value=df['Country'].unique().tolist(),
        multi=True
    ),

    html.Br(),

    # KPI CARDS
    html.Div([

        html.Div([
            html.H3("Total Sales"),
            html.H2(id='sales_card')
        ], style={
            'width': '30%',
            'display': 'inline-block',
            'backgroundColor': '#D6E4FF',
            'padding': '20px',
            'borderRadius': '10px',
            'textAlign': 'center'
        }),

        html.Div([
            html.H3("Total Orders"),
            html.H2(id='orders_card')
        ], style={
            'width': '30%',
            'display': 'inline-block',
            'backgroundColor': '#E3FCEF',
            'padding': '20px',
            'borderRadius': '10px',
            'marginLeft': '20px',
            'textAlign': 'center'
        }),

    ]),

    html.Br(),

    # CHARTS
    dcc.Graph(id='sales_chart'),

    dcc.Graph(id='product_chart'),

    dcc.Graph(id='country_chart')

])

In [ ]:
@app.callback(

    [
        Output('sales_card', 'children'),
        Output('orders_card', 'children'),
        Output('sales_chart', 'figure'),
        Output('product_chart', 'figure'),
        Output('country_chart', 'figure')
    ],

    [
        Input('country_filter', 'value')
    ]

)

def update_dashboard(selected_country):

    filtered_df = df[df['Country'].isin(selected_country)]

    # KPI VALUES
    total_sales = f"₹ {filtered_df['TotalPrice'].sum():,.0f}"

    total_orders = filtered_df['InvoiceNo'].nunique()

    # MONTHLY SALES
    monthly_sales = (
        filtered_df.groupby('Month')['TotalPrice']
        .sum()
        .reset_index()
    )

    fig1 = px.line(
        monthly_sales,
        x='Month',
        y='TotalPrice',
        title='Monthly Sales Trend',
        markers=True
    )

    # TOP PRODUCTS
    top_products = (
        filtered_df.groupby('Description')['TotalPrice']
        .sum()
        .sort_values(ascending=False)
        .head(10)
        .reset_index()
    )

    fig2 = px.bar(
        top_products,
        x='TotalPrice',
        y='Description',
        orientation='h',
        title='Top 10 Products'
    )

    # COUNTRY SALES
    country_sales = (
        filtered_df.groupby('Country')['TotalPrice']
        .sum()
        .reset_index()
    )

    fig3 = px.pie(
        country_sales,
        names='Country',
        values='TotalPrice',
        title='Country-wise Sales'
    )

    return (
        total_sales,
        total_orders,
        fig1,
        fig2,
        fig3
    )

In [ ]:
app.run(debug=False)